# Analysis of experimental data

This notebook documents analysis of experimental data resulting from developed pitch pipelines. This document is focused on analysis and findings. For complete analysis code see ./pitch_analysis.py

## Datasets used

- [ChoraleBricks](https://zenodo.org/records/15081741)

## Metrics

### GPE (Gross Pitch Error)
The proportion of (voiced) frames for which estimated f0 was substantially different from the reference data

### MAE (Mean Absolute Error)
The average absolute error across all frames

In [6]:
from pathlib import Path
import pandas as pd
import pitch_analysis

Here we load the raw data from experiments conducted in kotlin, including timestamped predictions per dataset per file, but without labelled f0 data.

In [ ]:
window = "1024"
filename = f"ChoraleBricks__windowSize{window}.csv"
summary_export_name = Path(".") / "results" / "effect_of_window_size" / f"{window}_window.csv"

results_path = Path(".") / "experimental_data" / filename
experiment_data = pitch_analysis.PitchExperimentData(results_path)
experiment_data.head()

,dataset,file,estimator,timestamp,f0,processing_time_file
0,ChoraleBricks,Anonymous_AusMeinesHerzensGrunde__01_as.wav,YinEstimator,0.03483,-1.0,534.0
1,ChoraleBricks,Anonymous_AusMeinesHerzensGrunde__01_as.wav,YinEstimator,0.05805,-1.0,534.0
2,ChoraleBricks,Anonymous_AusMeinesHerzensGrunde__01_as.wav,YinEstimator,0.08127,-1.0,534.0
3,ChoraleBricks,Anonymous_AusMeinesHerzensGrunde__01_as.wav,YinEstimator,0.10449,-1.0,534.0
4,ChoraleBricks,Anonymous_AusMeinesHerzensGrunde__01_as.wav,YinEstimator,0.12771,-1.0,534.0


# Data processing
Here we conduct the following steps on the raw experimental data:

1. For each audio file, find the corresponding labelled data
2. Align the estimated timestamps with the labelled data
3. Based on aligned timestamps, compare labelled f0 vs predicted f0 across dataset to calculate metrics

In [8]:
processed_data = experiment_data.process()
processed_data

,dataset,track,estimator,MAE,GPE,processing_time_file,instrument
0,ChoraleBricks,Anonymous_AusMeinesHerzensGrunde__01_as.wav,YinEstimator,17.898144,4.062240,534.0,as
1,ChoraleBricks,Anonymous_AusMeinesHerzensGrunde__01_as.wav,DynamicWaveletEstimator,22.213817,3.930393,881.0,as
2,ChoraleBricks,Anonymous_AusMeinesHerzensGrunde__01_as.wav,McLeodEstimator,20.118523,3.235048,2336.0,as
3,ChoraleBricks,Anonymous_AusMeinesHerzensGrunde__01_bar.wav,YinEstimator,10.462289,3.105665,176.0,bar
4,ChoraleBricks,Anonymous_AusMeinesHerzensGrunde__01_bar.wav,DynamicWaveletEstimator,62.774898,11.561383,377.0,bar
...,...,...,...,...,...,...,...
574,ChoraleBricks,Vulpius_DieHelleSonnLeuchtJetztHerfuer__04_bs.wav,DynamicWaveletEstimator,245.627180,17.718447,411.0,bs
575,ChoraleBricks,Vulpius_DieHelleSonnLeuchtJetztHerfuer__04_bs.wav,McLeodEstimator,28.888969,4.731183,1871.0,bs
576,ChoraleBricks,Vulpius_DieHelleSonnLeuchtJetztHerfuer__04_tba...,YinEstimator,23.403413,10.486177,74.0,tba
577,ChoraleBricks,Vulpius_DieHelleSonnLeuchtJetztHerfuer__04_tba...,DynamicWaveletEstimator,188.641032,19.406151,142.0,tba


In [9]:
# add a column for instrument
processed_data["instrument"] =    (
        processed_data["track"]
            .str
            .split("_")
            .str[-1]
            .str
            .removesuffix(".wav")
)
processed_data

,dataset,track,estimator,MAE,GPE,processing_time_file,instrument
0,ChoraleBricks,Anonymous_AusMeinesHerzensGrunde__01_as.wav,YinEstimator,17.898144,4.062240,534.0,as
1,ChoraleBricks,Anonymous_AusMeinesHerzensGrunde__01_as.wav,DynamicWaveletEstimator,22.213817,3.930393,881.0,as
2,ChoraleBricks,Anonymous_AusMeinesHerzensGrunde__01_as.wav,McLeodEstimator,20.118523,3.235048,2336.0,as
3,ChoraleBricks,Anonymous_AusMeinesHerzensGrunde__01_bar.wav,YinEstimator,10.462289,3.105665,176.0,bar
4,ChoraleBricks,Anonymous_AusMeinesHerzensGrunde__01_bar.wav,DynamicWaveletEstimator,62.774898,11.561383,377.0,bar
...,...,...,...,...,...,...,...
574,ChoraleBricks,Vulpius_DieHelleSonnLeuchtJetztHerfuer__04_bs.wav,DynamicWaveletEstimator,245.627180,17.718447,411.0,bs
575,ChoraleBricks,Vulpius_DieHelleSonnLeuchtJetztHerfuer__04_bs.wav,McLeodEstimator,28.888969,4.731183,1871.0,bs
576,ChoraleBricks,Vulpius_DieHelleSonnLeuchtJetztHerfuer__04_tba...,YinEstimator,23.403413,10.486177,74.0,tba
577,ChoraleBricks,Vulpius_DieHelleSonnLeuchtJetztHerfuer__04_tba...,DynamicWaveletEstimator,188.641032,19.406151,142.0,tba


In [10]:
mae_by_instrument = (
    processed_data.groupby("instrument", as_index=False)["MAE"]
      .mean()
)
mae_by_dataset = (
    processed_data.groupby("dataset",as_index=False)["MAE"].mean()
)

agg = pd.DataFrame(processed_data)

mae_by_dataset


summary = (
    processed_data.groupby(["estimator", "dataset"], as_index=False)
      .agg({
          "MAE": "mean",
          "GPE": "mean"
      })
)

experiment_data.summary().to_csv(summary_export_name)
experiment_data.summary()

,estimator,dataset,MAE,GPE,processing_time_file
0,DynamicWaveletEstimator,ChoraleBricks,108.377856,10.694251,388.461140
1,McLeodEstimator,ChoraleBricks,17.450212,3.438960,2606.165803
2,YinEstimator,ChoraleBricks,40.023509,5.818493,261.393782
